# 🌲 Priority Queues — Heaps e Implementación con `HeapPriorityQueue`

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap09/02_Heaps_Implementacion_Teoria.ipynb)

### 🎯 Objetivos

1. ✅ Comprender la estructura **heap** y sus propiedades fundamentales
2. ✅ Implementar `HeapPriorityQueue` basada en array con **up-heap** y **down-heap**
3. ✅ Analizar la complejidad O(log n) de add y remove_min
4. ✅ Usar el módulo **`heapq`** de Python para problemas prácticos
5. ✅ Entender la construcción **bottom-up** de un heap en O(n)

### 📋 Contenido

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 9.3)  

| Sección | Tema | Tiempo |
|---------|------|--------|
| 9.3.1 | Estructura y propiedades del Heap | 20 min |
| 9.3.2 | Representación con Array | 15 min |
| 9.3.3 | `HeapPriorityQueue` completa | 40 min |
| 9.3.4 | Bottom-up construction | 15 min |
| 9.3.6 | Módulo `heapq` de Python | 20 min |


---

## 📌 9.3.1 — La Estructura Heap

### Definición

Un **heap** es un árbol binario que satisface **dos propiedades**:

#### Propiedad 1: Heap-Order (Orden de Montículo)

> Para todo nodo `v` distinto de la raíz, la **clave de `v` ≥ clave de su padre**.

Consecuencia: el **mínimo** siempre está en la **raíz**.

#### Propiedad 2: Árbol Binario Completo

> Un árbol binario con `n` nodos y altura `h` es **completo** si los niveles
> `0, 1, ..., h-1` están llenos y en el nivel `h` los nodos están **rellenos de izquierda a derecha**.

```
          4              ← raíz (mínimo)
       /     \
      5        6
   /    \     /  \
  15     9   7   20
 / \    /
16  25 14
```

Esta es una heap válida (min-heap). Verificar:
- 5 ≥ 4 ✅, 6 ≥ 4 ✅, 15 ≥ 5 ✅, 9 ≥ 5 ✅, 7 ≥ 6 ✅, 20 ≥ 6 ✅, 16 ≥ 15 ✅, 25 ≥ 15 ✅, 14 ≥ 9 ✅
- Árbol binario completo ✅

### 📐 Proposición 9.2 — Altura del Heap

> Un heap que almacena `n` entradas tiene altura `h = ⌊log n⌋`.

| n | h = ⌊log₂ n⌋ |
|---|:------------:|
| 1 | 0 |
| 3 | 1 |
| 7 | 2 |
| 15 | 3 |
| 1023 | 9 |
| 1000000 | 19 |

Esta es la razón de la eficiencia O(log n) del heap.


---

## 📌 9.3.2 — Representación con Array

El árbol binario completo se representa eficientemente con un **array/lista**:

```
Árbol:        4
            /   \
          5       6
         / \     /
        15   9   7

Array: [4, 5, 6, 15, 9, 7]
índice: 0  1  2   3  4  5
```

### Fórmulas de Navegación

| Relación | Fórmula |
|----------|---------|
| Padre de nodo `j` | `(j - 1) // 2` |
| Hijo izquierdo de `j` | `2*j + 1` |
| Hijo derecho de `j` | `2*j + 2` |

### Verificación

- Padre del nodo 3 (=15): `(3-1)//2 = 1` → nodo 1 (=5) ✅
- Hijo izq del nodo 1 (=5): `2*1+1 = 3` → nodo 3 (=15) ✅
- Hijo der del nodo 1 (=5): `2*1+2 = 4` → nodo 4 (=9) ✅

Esta representación elimina la necesidad de punteros, siendo muy eficiente en memoria.


In [1]:
# Demostración de las fórmulas de navegación
heap_array = [4, 5, 6, 15, 9, 7, 20, 16, 25, 14]

def parent(j):      return (j - 1) // 2
def left(j):        return 2 * j + 1
def right(j):       return 2 * j + 2
def has_left(j, n): return left(j) < n
def has_right(j, n): return right(j) < n

n = len(heap_array)
print('Heap array:', heap_array)
print()
print(f'{"Nodo j":>8} | {"Valor":>6} | {"Padre idx":>10} | {"Padre val":>10} | {"IzqIdx":>8} | {"IzqVal":>8} | {"DerIdx":>8} | {"DerVal":>8}')
print('-' * 90)
for j in range(n):
    p = parent(j) if j > 0 else '-'
    pv = heap_array[p] if j > 0 else '-'
    li = left(j) if has_left(j, n) else '-'
    lv = heap_array[li] if has_left(j, n) else '-'
    ri = right(j) if has_right(j, n) else '-'
    rv = heap_array[ri] if has_right(j, n) else '-'
    print(f'{j:>8} | {heap_array[j]:>6} | {str(p):>10} | {str(pv):>10} | {str(li):>8} | {str(lv):>8} | {str(ri):>8} | {str(rv):>8}')


Heap array: [4, 5, 6, 15, 9, 7, 20, 16, 25, 14]

  Nodo j |  Valor |  Padre idx |  Padre val |   IzqIdx |   IzqVal |   DerIdx |   DerVal
------------------------------------------------------------------------------------------
       0 |      4 |          - |          - |        1 |        5 |        2 |        6
       1 |      5 |          0 |          4 |        3 |       15 |        4 |        9
       2 |      6 |          0 |          4 |        5 |        7 |        6 |       20
       3 |     15 |          1 |          5 |        7 |       16 |        8 |       25
       4 |      9 |          1 |          5 |        9 |       14 |        - |        -
       5 |      7 |          2 |          6 |        - |        - |        - |        -
       6 |     20 |          2 |          6 |        - |        - |        - |        -
       7 |     16 |          3 |         15 |        - |        - |        - |        -
       8 |     25 |          3 |         15 |        - |        - | 

---

## 📌 9.3.3 — `HeapPriorityQueue`: Operación `add`

### Up-Heap Bubbling (Subir burbujeando)

Después de insertar un nuevo elemento al **final** del array:

1. Comparar con su padre
2. Si `hijo < padre` → **intercambiar** (swap)
3. Repetir hacia arriba hasta llegar a la raíz o que el orden sea correcto

```
Antes de add(2, 'Z'):          Insertar al final:
        4                              4
      /   \                          /   \
    5       6                      5       6
   / \     / \                    / \     / \
  15  9   7  20                  15  9   7  20
                                /
                               2   ← insertado aquí (j=8)

Up-heap: 2 < 15 → swap(8,3):   Up-heap: 2 < 5 → swap(3,1):   2 < 4 → swap(1,0):
        4                              4                              2
      /   \                          /   \                          /   \
    5       6                      2       6                      4       6
   / \     / \                    / \     / \                    / \     / \
  2  9   7  20                  5   9   7  20                  5   9   7  20
 /                             /                              /
15                            15                             15
```

**Complejidad:** O(log n) — la altura máxima del árbol.


---

## 📌 9.3.3 — `HeapPriorityQueue`: Operación `remove_min`

### Down-Heap Bubbling (Bajar burbujeando)

1. **Guardar** el elemento raíz (mínimo)
2. **Mover** el último elemento del array a la raíz
3. **Comparar** la nueva raíz con sus hijos
4. **Intercambiar** con el hijo de **menor clave**
5. Repetir hacia abajo hasta ser hoja o estar en la posición correcta

```
Antes:                  Mover último a raíz:     Down-heap: swap(0,1):
        4                      14                      5
      /   \                  /   \                   /   \
    5       6              5       6               14     6
   / \     / \            / \     /               / \     /
  15  9   7  14          15  9   7               15  9   7
                        ← 14 se saca aquí

Down-heap: 14 > min(15,9) → swap(1,4):
        5
      /   \
    9       6
   / \     /
  15  14  7
```

**Complejidad:** O(log n)


In [2]:
class Empty(Exception):
    pass


class PriorityQueueBase:
    class _Item:
        __slots__ = '_key', '_value'
        def __init__(self, k, v):
            self._key = k
            self._value = v
        def __lt__(self, other):
            return self._key < other._key
        def __repr__(self):
            return f'({self._key},{self._value!r})'

    def is_empty(self):
        return len(self) == 0


class HeapPriorityQueue(PriorityQueueBase):
    """A min-oriented priority queue implemented with a binary heap.
    
    Complejidad:
        add(k,v)       → O(log n)
        min()          → O(1)
        remove_min()   → O(log n)
    """

    # ----------------------- nonpublic behaviors -----------------------
    def _parent(self, j):      return (j - 1) // 2
    def _left(self, j):        return 2 * j + 1
    def _right(self, j):       return 2 * j + 2
    def _has_left(self, j):    return self._left(j) < len(self._data)
    def _has_right(self, j):   return self._right(j) < len(self._data)

    def _swap(self, i, j):
        """Swap the elements at indices i and j of array."""
        self._data[i], self._data[j] = self._data[j], self._data[i]

    def _upheap(self, j):
        """Move newly added item at j upward to restore heap-order."""
        parent = self._parent(j)
        if j > 0 and self._data[j] < self._data[parent]:
            self._swap(j, parent)
            self._upheap(parent)    # recur at position of parent

    def _downheap(self, j):
        """Move item at index j downward to restore heap-order."""
        if self._has_left(j):
            left = self._left(j)
            small_child = left    # aunque no tenga derecho
            if self._has_right(j):
                right = self._right(j)
                if self._data[right] < self._data[left]:
                    small_child = right
            if self._data[small_child] < self._data[j]:
                self._swap(j, small_child)
                self._downheap(small_child)    # recur

    # ----------------------- public behaviors -----------------------
    def __init__(self, contents=()):
        """Create a new priority queue.
        
        By default, queue will be empty. If contents is given, it should be
        an iterable of (k,v) tuples specifying the initial contents.
        """
        self._data = [self._Item(k, v) for k, v in contents]
        if len(self._data) > 1:
            self._heapify()

    def _heapify(self):
        """Bottom-up heap construction. O(n) time."""
        start = self._parent(len(self._data) - 1)   # índice del último nodo interno
        for j in range(start, -1, -1):              # recorrer hacia la raíz
            self._downheap(j)

    def __len__(self):
        return len(self._data)

    def min(self):
        """Return but do not remove (k,v) with minimum key. O(1)."""
        if self.is_empty():
            raise Empty('Priority queue is empty')
        item = self._data[0]
        return (item._key, item._value)

    def add(self, key, value):
        """Add a key-value pair to the priority queue. O(log n)."""
        self._data.append(self._Item(key, value))
        self._upheap(len(self._data) - 1)

    def remove_min(self):
        """Remove and return (k,v) tuple with minimum key. O(log n)."""
        if self.is_empty():
            raise Empty('Priority queue is empty')
        self._swap(0, len(self._data) - 1)   # poner mínimo al final
        item = self._data.pop()               # y sacarlo
        self._downheap(0)                     # restaurar heap-order
        return (item._key, item._value)

    def __repr__(self):
        return f'HeapPQ({self._data})'


# ── Demostración ──
print('=== HeapPriorityQueue — Demo ===')
H = HeapPriorityQueue()
for k, v in [(5,'A'), (9,'C'), (3,'B'), (7,'D'), (1,'F'), (8,'E')]:
    H.add(k, v)
    print(f'add({k},{v!r}) → heap: {H._data}')

print()
print(f'min()        → {H.min()}')
print(f'remove_min() → {H.remove_min()} | heap: {H._data}')
print(f'remove_min() → {H.remove_min()} | heap: {H._data}')
print(f'remove_min() → {H.remove_min()} | heap: {H._data}')
print(f'len(H)       → {len(H)}')


=== HeapPriorityQueue — Demo ===
add(5,'A') → heap: [(5,'A')]
add(9,'C') → heap: [(5,'A'), (9,'C')]
add(3,'B') → heap: [(3,'B'), (9,'C'), (5,'A')]
add(7,'D') → heap: [(3,'B'), (7,'D'), (5,'A'), (9,'C')]
add(1,'F') → heap: [(1,'F'), (3,'B'), (5,'A'), (9,'C'), (7,'D')]
add(8,'E') → heap: [(1,'F'), (3,'B'), (5,'A'), (9,'C'), (7,'D'), (8,'E')]

min()        → (1, 'F')
remove_min() → (1, 'F') | heap: [(3,'B'), (7,'D'), (5,'A'), (9,'C'), (8,'E')]
remove_min() → (3, 'B') | heap: [(5,'A'), (7,'D'), (8,'E'), (9,'C')]
remove_min() → (5, 'A') | heap: [(7,'D'), (9,'C'), (8,'E')]
len(H)       → 3


---

## 📌 9.3.4 — Construcción Bottom-Up del Heap

### ¿Por qué O(n)?

Insertar n elementos uno a uno cuesta **O(n log n)**.  
Pero si conocemos todos los elementos de antemano, podemos construir el heap en **O(n)**:

### Algoritmo

1. Cargar todos los n elementos en el array (sin orden)
2. Comenzar desde el **último nodo interno** (índice `(n-1)//2 - 1` ≈ `n//2 - 1`)
3. Aplicar `_downheap` hacia la raíz

```python
def _heapify(self):
    start = self._parent(len(self._data) - 1)
    for j in range(start, -1, -1):
        self._downheap(j)
```

### Análisis

- Los nodos hoja ya satisfacen heap-order (sin hijos)
- Trabajamos de abajo hacia arriba: nodos con subárboles pequeños primero
- Costo amortizado total: O(n) operaciones de swap

> **Proposición 9.3:** La construcción bottom-up de un heap con n entradas
> se puede realizar en O(n) tiempo.


In [ ]:
import time

# Construir heap con n elementos dados de antemano (O(n))
datos = [(5,'A'), (9,'C'), (3,'B'), (7,'D'), (1,'F'), (8,'E'), (2,'G'), (6,'H')]

print('=== Construcción Bottom-Up (O(n)) ===')
H_fast = HeapPriorityQueue(contents=datos)
print(f'Heap construida de una vez: {H_fast._data}')
print(f'min() = {H_fast.min()}')
print()

# Comparar tiempos: bottom-up vs inserción individual
N = 100_000

# Bottom-up: O(n)
t0 = time.perf_counter()
h_bu = HeapPriorityQueue(contents=[(i, f'v{i}') for i in range(N, 0, -1)])
t_bu = time.perf_counter() - t0

# Individual: O(n log n)
t0 = time.perf_counter()
h_ind = HeapPriorityQueue()
for i in range(N, 0, -1):
    h_ind.add(i, f'v{i}')
t_ind = time.perf_counter() - t0

print(f'N = {N:,} elementos:')
print(f'  Bottom-up O(n):       {t_bu*1000:.2f} ms')
print(f'  Individual O(n logN): {t_ind*1000:.2f} ms')
print(f'  Speedup: {t_ind/t_bu:.2f}x')


---

## 📌 9.3.6 — El Módulo `heapq` de Python

Python incluye un módulo estándar `heapq` que implementa un **min-heap** sobre una lista ordinaria:

### Funciones Principales

| Función | Descripción | Complejidad |
|---------|-------------|-------------|
| `heapq.heappush(h, item)` | Inserta `item` en el heap `h` | O(log n) |
| `heapq.heappop(h)` | Extrae y retorna el mínimo | O(log n) |
| `heapq.heapify(h)` | Convierte lista en heap in-place | O(n) |
| `heapq.heappushpop(h, item)` | Push + pop combinado (más eficiente) | O(log n) |
| `heapq.heapreplace(h, item)` | Pop + push combinado | O(log n) |
| `heapq.nlargest(n, iterable)` | Los n más grandes | O(n + k log n) |
| `heapq.nsmallest(n, iterable)` | Los n más pequeños | O(n + k log n) |

> **⚠️ Importante:** `heapq` es un **min-heap**. Para max-heap, negar los valores.


In [ ]:
import heapq

print('=== Módulo heapq de Python ===')

# 1. heappush / heappop
h = []
for k in [5, 3, 9, 1, 7, 4]:
    heapq.heappush(h, k)
    print(f'push({k:2d}) → heap: {h}')

print()
print('Extrayendo en orden:')
while h:
    print(f'  heappop() → {heapq.heappop(h)} | resto: {h}')

print()

# 2. heapify (bottom-up in-place)
data = [8, 3, 5, 1, 9, 2, 7, 4, 6]
print(f'Antes heapify: {data}')
heapq.heapify(data)
print(f'Después heapify: {data}')

print()

# 3. nlargest / nsmallest
notas = [85, 92, 78, 95, 88, 72, 99, 81, 90, 76]
print(f'Notas: {notas}')
print(f'Top 3 notas más altas: {heapq.nlargest(3, notas)}')
print(f'Top 3 notas más bajas: {heapq.nsmallest(3, notas)}')

print()

# 4. heapq con tuplas (clave, valor) — como Priority Queue
print('=== heapq como Priority Queue ===')
pq = []
heapq.heappush(pq, (5, 'tarea regular'))
heapq.heappush(pq, (1, 'tarea urgente'))
heapq.heappush(pq, (3, 'tarea importante'))
heapq.heappush(pq, (1, 'otra tarea urgente'))  # misma prioridad

while pq:
    priority, task = heapq.heappop(pq)
    print(f'  Procesando (prioridad={priority}): {task}')


In [ ]:
import heapq

print('=== Max-Heap con heapq (negando valores) ===')

# Python solo tiene min-heap; para max-heap negamos los valores
numeros = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
max_heap = []

for n in numeros:
    heapq.heappush(max_heap, -n)  # Insertar negado

print('Extrayendo en orden DESCENDENTE:')
resultado = []
while max_heap:
    resultado.append(-heapq.heappop(max_heap))  # Negar al extraer

print(f'  {resultado}')
print(f'  ¿Descendente? {resultado == sorted(numeros, reverse=True)}')

# Aplicación práctica: k-ésimo mayor elemento
def kth_largest(nums, k):
    """Encontrar el k-ésimo elemento más grande. O(n log k)."""
    h = []
    for n in nums:
        heapq.heappush(h, n)
        if len(h) > k:
            heapq.heappop(h)   # Mantener solo los k mayores
    return h[0]   # El mínimo del heap de los k mayores = k-ésimo mayor

datos = [3, 2, 1, 5, 6, 4]
for k in range(1, 5):
    print(f'{k}-ésimo mayor de {datos} = {kth_largest(datos, k)}')


---

## 📝 Resumen del Notebook 2

### La Estructura Heap

| Propiedad | Descripción |
|-----------|-------------|
| **Heap-Order** | Clave de todo nodo ≥ clave de su padre |
| **Árbol Completo** | Todos los niveles completos excepto el último (izq a der) |
| **Altura** | h = ⌊log n⌋ → eficiencia O(log n) |

### Complejidad de `HeapPriorityQueue`

| Operación | Complejidad |
|-----------|-------------|
| `len()`, `is_empty()` | O(1) |
| `min()` | **O(1)** |
| `add(k,v)` | **O(log n)** |
| `remove_min()` | **O(log n)** |
| Construcción (n elementos) | **O(n)** |

### `heapq` de Python

- Es un min-heap implementado sobre una lista Python
- Para max-heap: negar los valores
- `heapify` para construcción eficiente en O(n)
- `nlargest`/`nsmallest` para top-k queries

---

🔗 **Referencia:** Goodrich et al., Cap. 9.3

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>
